# 🛡️ SecureBERT URL Phishing Classifier — Google Colab
**Model:** SecureBERT (BERT pre-trained on cybersecurity text)  
**Output:** `url_classifier.zip` → download → drop into your app folder  

### ⚡ Before running:
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save  
2. Then `Runtime` → `Run all`

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    lines = result.stdout.split('\n')
    for line in lines[:15]:
        print(line)
    print("\n✅ GPU is active!")
else:
    print("❌ NO GPU DETECTED!")
    print("Go to: Runtime → Change runtime type → T4 GPU → Save")
    print("Then Runtime → Run all")
    raise SystemExit("Enable GPU first!")

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys
packages = ['transformers==4.40.0', 'torch', 'scikit-learn',
            'tldextract', 'pandas', 'joblib', 'accelerate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('✅ All packages installed')

## Step 2 — Upload Dataset CSV
Download the dataset from:  
👉 https://kaggle.com/datasets/harisudhan411/phishing-and-legitimate-urls  
Click **Download** → unzip → you'll get a `.csv` file → upload it below

In [ ]:
from google.colab import files
import pandas as pd
import io

print("📂 Click 'Choose Files' and select the CSV from the dataset...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"\nUploaded: {filename}")
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(df_raw.head(3))

## Step 3 — Normalize & Balance Dataset

In [ ]:
# Auto-detect url and label columns
col_map = {}
for c in df_raw.columns:
    cl = c.lower().strip()
    if cl in ['url','urls','link','domain']:   col_map[c] = 'url'
    elif cl in ['label','type','status','class','result','tag']: col_map[c] = 'label'

if 'url' not in col_map.values() or 'label' not in col_map.values():
    print("⚠️ Could not auto-detect columns. Available:", df_raw.columns.tolist())
    # Manually set if needed:
    # col_map = {'URL_column_name': 'url', 'Label_column_name': 'label'}

df = df_raw.rename(columns=col_map)[['url','label']].dropna()

# Normalize labels to 0/1
label_map = {
    'bad':1,'phishing':1,'malicious':1,'spam':1,'1':1,1:1,
    'good':0,'benign':0,'legitimate':0,'safe':0,'0':0,0:0
}
df['label'] = df['label'].map(label_map)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"After cleaning → Total: {len(df):,}")
print(f"  Phishing  : {(df.label==1).sum():,}")
print(f"  Legitimate: {(df.label==0).sum():,}")

# Balance — 80k each
phish = df[df.label==1]
legit = df[df.label==0]
n = min(len(phish), len(legit), 80_000)
df_bal = pd.concat([
    phish.sample(n, random_state=42),
    legit.sample(n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset: {len(df_bal):,} rows ({n:,} per class)")

## Step 4 — Preprocess URLs for SecureBERT

In [ ]:
import re, unicodedata

def url_to_text(url: str) -> str:
    """
    Convert URL to readable tokens so BERT understands meaning.
    
    Examples:
      http://paypal-secure-verify.login.tk  →  'paypal secure verify login tk'
      https://google.com/search?q=test      →  'google com search test'
      http://amaz0n-acc0unt.update.xyz      →  'amaz0n acc0unt update xyz'
    
    BERT then reads these as words and uses its cybersecurity knowledge
    to judge if the combination looks phishing-like.
    """
    url = str(url).lower().strip()
    url = re.sub(r'https?://', '', url)
    url = re.sub(r'^www\.', '', url)
    url = unicodedata.normalize('NFKD', url).encode('ascii','ignore').decode('ascii')
    tokens = re.split(r'[^a-z0-9]', url)
    tokens = [t for t in tokens if t and len(t) > 1]
    return ' '.join(tokens[:50])  # max 50 tokens

# Show examples
print("URL → Text conversion examples:")
print("-" * 65)
examples = [
    "http://paypal-secure-verify.login.tk/account/update",
    "https://google.com/search?q=python",
    "http://192.168.1.1/admin/login",
    "http://amaz0n-acc0unt.update-secure.xyz/signin",
    "https://elearning.univ-boumerdes.dz",
    "http://phishing-site.example",
]
for u in examples:
    print(f"  IN : {u}")
    print(f"  OUT: {url_to_text(u)}")
    print()

df_bal['text'] = df_bal['url'].apply(url_to_text)
print(f"✅ Preprocessed {len(df_bal):,} URLs")

## Step 5 — Load SecureBERT & Tokenize

In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

MODEL_NAME = "ehsanaghaei/SecureBERT"
print(f"Loading SecureBERT tokenizer from HuggingFace...")
print("(First time downloads ~500MB — takes 1-2 min)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer loaded")

# Split
train_df, temp_df = train_test_split(df_bal, test_size=0.20,
                                     random_state=42, stratify=df_bal['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.50,
                                     random_state=42, stratify=temp_df['label'])

print(f"\nSplit → Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

def tokenize_batch(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

print("Tokenizing (may take 1-2 min)...")
train_enc = tokenize_batch(train_df['text'])
val_enc   = tokenize_batch(val_df['text'])
test_enc  = tokenize_batch(test_df['text'])
print("✅ Tokenization complete")

## Step 6 — Create PyTorch DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class URLDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels.values, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = URLDataset(train_enc, train_df['label'].reset_index(drop=True))
val_ds   = URLDataset(val_enc,   val_df['label'].reset_index(drop=True))
test_ds  = URLDataset(test_enc,  test_df['label'].reset_index(drop=True))

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## Step 7 — Fine-tune SecureBERT (~25-35 min on T4)

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import os, time

print(f"Loading SecureBERT model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
).to(device)
print("✅ Model loaded")

EPOCHS      = 3
optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = total_steps // 10,
    num_training_steps = total_steps
)

SAVE_DIR = '/content/url_classifier'
os.makedirs(SAVE_DIR, exist_ok=True)

def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for batch in loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            out    = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss_sum += out.loss.item()
            preds     = out.logits.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)
    return correct/total, loss_sum/len(loader)

best_val_acc = 0.0
print("\n🚀 Training started...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for i, batch in enumerate(train_loader):
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        out  = model(input_ids=ids, attention_mask=mask, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if i % 100 == 0:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch+1}/{EPOCHS} | Step {i:>4}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f} | Elapsed: {elapsed:.0f}s")

    val_acc, val_loss = evaluate(val_loader)
    print(f"\n✅ Epoch {epoch+1} done | Train Loss: {total_loss/len(train_loader):.4f} "
          f"| Val Acc: {val_acc:.4%} | Val Loss: {val_loss:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"  💾 Best model saved! (val_acc={val_acc:.4%})\n")
    else:
        print()

print(f"🏆 Training complete! Best val accuracy: {best_val_acc:.4%}")

## Step 8 — Final Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Load best saved model
from transformers import AutoModelForSequenceClassification
best_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        out    = best_model(input_ids=ids, attention_mask=mask)
        preds  = out.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].numpy())

print("=" * 50)
print("       FINAL TEST RESULTS")
print("=" * 50)
print(classification_report(all_labels, all_preds,
      target_names=['Legitimate','Phishing'], digits=4))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Legit','Phishing'])
ax.set_yticklabels(['Legit','Phishing'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — SecureBERT URL Classifier')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=13)
plt.tight_layout()
plt.show()

## Step 9 — Sanity Test on Real URLs

In [ ]:
test_urls = [
    ("https://google.com",                              "✅ legit"),
    ("https://elearning.univ-boumerdes.dz",             "✅ legit"),
    ("https://github.com/user/repo/tree/main",          "✅ legit"),
    ("https://stackoverflow.com/questions/123456",      "✅ legit"),
    ("http://phishing-site.example",                    "🚨 phishing"),
    ("http://paypal-secure-verify.login.tk",            "🚨 phishing"),
    ("http://192.168.1.1/admin/login",                  "🚨 phishing"),
    ("http://amaz0n-account.update-secure.xyz",         "🚨 phishing"),
    ("http://microsoft-account-verify.secure-login.tk", "🚨 phishing"),
    ("http://facebook.com.phishing-login.ru",           "🚨 phishing"),
]

best_model.eval()
print(f"{'URL':<57} {'BERT%':>6}  Result         Expected")
print("-" * 100)

all_correct = 0
with torch.no_grad():
    for url, expected in test_urls:
        text   = url_to_text(url)
        enc    = tokenizer(text, return_tensors='pt', truncation=True,
                           max_length=128, padding='max_length').to(device)
        logits = best_model(**enc).logits
        probs  = torch.softmax(logits, dim=1)[0]
        score  = probs[1].item()
        label  = '🚨 PHISHING' if score > 0.5 else '✅ LEGIT   '
        correct = '✓' if ('phishing' in expected) == (score > 0.5) else '✗ WRONG'
        if correct == '✓': all_correct += 1
        print(f"{url:<57} {score:>6.2%}  {label}  {expected}  {correct}")

print(f"\nScore: {all_correct}/{len(test_urls)} correct")

## Step 10 — Download Model

In [ ]:
import shutil, os
from google.colab import files

# Show what's being saved
print("Files in url_classifier/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024 / 1024
    print(f"  {f:<45} {size:.1f} MB")

# Zip and download
zip_path = '/content/url_classifier'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
total_mb = os.path.getsize(zip_path + '.zip') / 1024 / 1024
print(f"\n📦 Zipped: {total_mb:.0f} MB")
print("⬇️  Starting download...")
files.download('/content/url_classifier.zip')
print("\n✅ Done! After downloading:")
print("   1. Unzip url_classifier.zip")
print("   2. Put the url_classifier/ folder next to app.py")
print("   3. pip install transformers torch")
print("   4. python app.py")